In [1]:
import io
import gc
import pymupdf as fitz
import torch
import base64
import transformers
from PIL import Image
import pypdfium2 as pdfium
from dotenv import load_dotenv

from huggingface_hub.utils import enable_progress_bars
from colpali_engine.models import ColQwen2, ColQwen2Processor

from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions, AcceleratorOptions, AcceleratorDevice
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption
from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend

c:\Users\UserAdmin\Documents\Multimodal-RAG\.3.12-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
load_dotenv()

True

In [6]:
pdf_path = r"C:\Users\UserAdmin\Documents\Multimodal-RAG\pdfs\government-personal-data-protection-policies-jul21.pdf"

In [7]:
model_name = "vidore/colqwen2-v1.0"
model = ColQwen2.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
).eval()

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 3178.13it/s]


In [8]:
processor = ColQwen2Processor.from_pretrained(
    model_name, 
    trust_remote_code=True
)

In [2]:
def get_coarse_embedding(embeddings_tensor):
    pooled = embeddings_tensor.mean(dim=1) # mean pooling 
    normalised = pooled / pooled.norm(dim=-1, keepdim=True) # L2 normalization 
    return normalised.squeeze().to(torch.float32).cpu()

In [ ]:
def embed(pdf_path, model, processor):
    vector_index = []
    fitz_doc = fitz.open(pdf_path)
    for i in range(len(fitz_doc)):
        page = fitz_doc.load_page(i)
        pix = page.get_pixmap(dpi=150)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

        inputs = processor.process_images(images=[img])
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            embeddings = model(**inputs)
            embeddings = embeddings / embeddings.norm(dim=-1, keepdim=True) #L2 normalization 
            coarse_vector = get_coarse_embedding(embeddings).flatten().tolist()
            embeddings = embeddings.squeeze(0).to(torch.float32).cpu()

        vector_index.append({
            "page_number": i+1,
            "coarse_vector": coarse_vector,
            "page_embeddings": embeddings
        })
    return vector_index

        

    
    

In [4]:
import torch
print("Is GPU available?:", torch.cuda.is_available())
print("Current Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


print(torch.__version__)      # If it ends in '+cpu', you have the wrong version.
print(torch.version.cuda)     # If this is None, CUDA support is not compiled in.


Is GPU available?: True
Current Device: NVIDIA RTX 5000 Ada Generation Laptop GPU
2.12.0+cu132
13.2


In [9]:
vector_index = embed(pdf_path, model, processor)
print(vector_index)

[{'page_number': 1, 'coarse_vector': [0.244140625, -0.01409912109375, 0.09130859375, 0.09912109375, 0.056396484375, -0.07958984375, -0.0810546875, 0.0284423828125, 0.014404296875, 0.10791015625, -0.140625, -0.035400390625, -0.017578125, 0.0047607421875, 0.12060546875, -0.01446533203125, 0.09765625, -0.03759765625, 0.02587890625, -0.0037994384765625, -0.033447265625, 0.1064453125, 0.0205078125, -0.1220703125, -0.0458984375, -0.09033203125, -0.0478515625, -0.05078125, -0.1435546875, -0.028564453125, -0.0150146484375, 0.053955078125, -0.1279296875, 0.06201171875, -0.062255859375, -0.052734375, -0.06982421875, 0.0966796875, -0.03125, 0.076171875, 0.0308837890625, -0.10009765625, 0.06884765625, -0.01263427734375, -0.1796875, 0.018310546875, 0.0262451171875, -0.013916015625, -0.1513671875, 0.06591796875, 0.0164794921875, -0.0172119140625, -0.1591796875, 0.03857421875, 0.083984375, -0.0791015625, 0.1298828125, -0.09130859375, 0.1923828125, -0.10791015625, 0.08984375, 0.058349609375, 0.0022430

In [10]:
vector_index[0]['page_embeddings']

tensor([[ 0.0204, -0.0879, -0.0723,  ...,  0.0520, -0.0020,  0.1094],
        [ 0.0189, -0.1074, -0.0259,  ...,  0.0762, -0.0444, -0.1396],
        [-0.0020, -0.0981, -0.0198,  ...,  0.0757, -0.0339, -0.1299],
        ...,
        [ 0.0574, -0.1108, -0.0674,  ...,  0.0603,  0.0036,  0.0703],
        [ 0.0332, -0.1167, -0.0464,  ...,  0.0447, -0.0474, -0.1230],
        [ 0.1328, -0.0752, -0.0008,  ..., -0.0262, -0.1035, -0.0557]])